# **Collab Mini Project Klasifikasi Berita hoax & non-hoax di indonesia**

# **A. Scraping Dataset**

**Instalasi Library Scraping**

In [ ]:
 !pip install requests beautifulsoup4

**Mount Google Drive & Setup Direktori**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # ikuti instruksi autentikasi
OUTPUT_DIR = "/content/drive/MyDrive/turnbackhoax_data"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUTPUT_DIR: /content/drive/MyDrive/turnbackhoax_data


**Pemeriksaan robots.txt (Compliance Scraping)**

In [ ]:
import requests
HEADERS = {"User-Agent": "ResearchBot/1.0 (+mailto:you@example.com)"}
r = requests.get("https://turnbackhoax.id/robots.txt", headers=HEADERS, timeout=20)
print("robots.txt status:", r.status_code)
print("--- robots.txt content ---")
print(r.text[:1000])  # tampilkan sebagian agar tidak terlalu panjang



robots.txt status: 200
--- robots.txt content ---
User-agent: *
Disallow: /wp-admin/
Allow: /wp-admin/admin-ajax.php

Sitemap: https://turnbackhoax.id/wp-sitemap.xml



**Scraping dan Ekstraksi Dataset Artikel Hoaks dari TurnBackHoax.id**

In [ ]:
import requests, time, csv, os, re, random
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import pandas as pd

BASE_SITE = "https://turnbackhoax.id"
PAGE_URL = BASE_SITE + "/page/{}/"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; ResearchBot/1.0; +mailto:you@example.com)"}

# OUTPUT: ganti kalau tidak mount Drive
OUTPUT_DIR = "/content/turnbackhoax_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
RAW_HTML_DIR = os.path.join(OUTPUT_DIR, "raw_html")
os.makedirs(RAW_HTML_DIR, exist_ok=True)

# Set berapa banyak halaman yang mau di-scan
MAX_PAGES = 30         # ubah sesuai kebutuhan (jangan terlalu besar)
DELAY_MIN = 1.0
DELAY_MAX = 2.0

session = requests.Session()
session.headers.update(HEADERS)

def safe_filename(s):
    s = re.sub(r'[^0-9a-zA-Z\-_.]', '_', s)
    return s[:200]

def get_soup_from_url(url):
    r = session.get(url, timeout=30)
    if r.status_code != 200:
        return None, r.status_code, r.text
    return BeautifulSoup(r.text, "lxml"), r.status_code, r.text

def get_article_links_from_page(page_num):
    url = PAGE_URL.format(page_num)
    soup, status, raw = get_soup_from_url(url)
    if soup is None:
        return None, status
    links = []
    # Heuristik: ambil <article> lalu anchor, fallback ke selectors populer
    for art in soup.find_all("article"):
        a = art.find("a", href=True)
        if a:
            href = urljoin(BASE_SITE, a['href'])
            if href.startswith(BASE_SITE) and href not in links:
                links.append(href)
    if not links:
        # fallback selectors
        for sel in ["h2.entry-title a", "h3.entry-title a", "a[rel='bookmark']"]:
            for a in soup.select(sel):
                href = urljoin(BASE_SITE, a.get("href"))
                if href.startswith(BASE_SITE) and href not in links:
                    links.append(href)
    return links, status

def extract_label_from_soup(soup):
    # Coba cari elemen yang eksplisit (class mengandung verdict/label), lalu heuristik teks.
    candidates = []
    for tag in soup.find_all(True):
        cls = " ".join(tag.get("class") or [])
        if "verdict" in cls.lower() or "label" in cls.lower() or "status" in cls.lower() or "hasil" in cls.lower():
            txt = tag.get_text(" ", strip=True)
            if txt:
                candidates.append(txt)
    # heuristic: cari kata kunci umum
    text_top = ""
    content_top = soup.get_text(" ", strip=True)[:800].lower()
    keywords = ["hoax", "salah", "fakta", "benar", "misleading", "disinformasi", "provokasi"]
    for kw in keywords:
        if kw in content_top:
            # ambil kalimat yang mengandung keyword
            m = re.search(r'([^.!\n]{0,120}\b' + re.escape(kw) + r'\b[^.!\n]{0,120})', content_top)
            if m:
                candidates.append(m.group(1).strip())
    # return shortest candidate (lebih mungkin label ringkas), atau empty string
    if candidates:
        candidates = sorted(set(candidates), key=lambda x: len(x))
        # trim to 250 chars
        return candidates[0][:250]
    return ""

def parse_article(url):
    soup, status, raw = get_soup_from_url(url)
    if soup is None:
        return None, status
    # Simpan raw HTML untuk reproducibility
    parsed = urlparse(url)
    fname = safe_filename(parsed.path.strip("/").replace("/", "_") or parsed.netloc)
    with open(os.path.join(RAW_HTML_DIR, fname + ".html"), "w", encoding="utf-8") as f:
        f.write(raw)

    # Title
    title_tag = soup.find("h1", class_="entry-title") or soup.find("h1")
    title = title_tag.get_text(" ", strip=True) if title_tag else ""

    # Date
    date_tag = soup.find("time", class_="entry-date") or soup.find("time")
    date = date_tag.get("datetime") if date_tag and date_tag.get("datetime") else (date_tag.get_text(" ", strip=True) if date_tag else "")

    # Content
    content_div = soup.find("div", class_=re.compile(r"(post-content|entry-content|td-post-content|post-body)")) or soup.find("article")
    if content_div:
        paragraphs = [p.get_text(" ", strip=True) for p in content_div.find_all("p")]
        content = "\n\n".join([p for p in paragraphs if p])
    else:
        content = soup.get_text(" ", strip=True)[:2000]

    # Label / verdict (heuristic)
    label = extract_label_from_soup(soup)

    return {
        "url": url,
        "title": title,
        "date": date,
        "label": label,
        "content": content
    }, 200

# === RUN SCRAPE ===
all_records = []
seen_urls = set()
for page in range(1, MAX_PAGES + 1):
    print(f"Scanning page {page} ...", end=" ")
    links_status = get_article_links_from_page(page)
    if links_status is None:
        print("failed or 404, stop.")
        break
    links, status = links_status
    if not links:
        print("no links (empty) — stop.")
        break
    print(f"found {len(links)} links")
    for link in links:
        if link in seen_urls:
            continue
        try:
            rec, code = parse_article(link)
            if rec:
                all_records.append(rec)
                seen_urls.add(link)
                print(" -", rec["title"][:80])
            else:
                print(" - failed parse:", link, code)
        except Exception as e:
            print(" - error parsing:", e)
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
    # jeda antar halaman
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

# Simpan CSV
df = pd.DataFrame(all_records)
CSV_PATH = os.path.join(OUTPUT_DIR, "turnbackhoax_data.csv")
df.to_csv(CSV_PATH, index=False)
print("Saved", len(df), "records to", CSV_PATH)
# Show preview
df.head(10)

Scanning page 1 ... found 20 links
 - [SALAH] KTP-el Dipasang GPS untuk Pantau Keberadaan Masyarakat
 - [SALAH] Jokowi Setuju RUU Perampasan Aset, Asalkan Tidak Menyasar Mantan Preside
 - [SALAH] Israel Ngambek dan Keluar dari Sidang PBB Ketika Hampir Semua Negara Sep
 - [PENIPUAN] PLN Tawarkan Listrik Gratis Lewat Akun TikTok
 - [SALAH] Ojol Demo Kenaikan Pertalite
 - [SALAH] Panglima NATO akan Buat Kerusuhan di Indonesia
 - [SALAH] Ojol Dilarang Gunakan Pertalite
 - [SALAH] Video Pengiriman 18.000 Personel TNI ke Aceh
 - [SALAH] Pom Bensin Shell Resmi Tutup dan Hengkang dari Indonesia pada 2026
 - [SALAH] Menteri ESDM Bahlil Lahadalia Jadi Tersangka Kasus Korupsi Migas
 - [PENIPUAN] Wapres Gibran Promosikan Program Motor Murah
 - [SALAH] Mobil Ahmad Sahroni Dikepung di Tengah Demo Mahasiswa
 - [SALAH] Gatot Nurmantyo Resmi Jadi Menko Polkam
 - [SALAH] Prabowo Ungkap Gatot Nurmantyo dan Rocky Gerung Masuk Kabinet Merah Puti
 - [SALAH] KPK Panggil Megawati Untuk Diperiksa
 - [PENIPUAN]

,url,title,date,label,content
0,https://turnbackhoax.id/2025/09/30/salah-ktp-e...,[SALAH] KTP-el Dipasang GPS untuk Pantau Keber...,,[salah] ktp-el dipasang gps untuk pantau keber...,Akun Instagram “ 1273alam ” [ arsip ] pada Kam...
1,https://turnbackhoax.id/2025/09/30/salah-jokow...,"[SALAH] Jokowi Setuju RUU Perampasan Aset, Asa...",,"[salah] jokowi setuju ruu perampasan aset, asa...",Pada Selasa (23/9/2025) akun Facebook “Himae H...
2,https://turnbackhoax.id/2025/09/30/salah-israe...,[SALAH] Israel Ngambek dan Keluar dari Sidang ...,,[salah] israel ngambek dan keluar dari sidang ...,Pada Selasa (23/09/2025) akun Tiktok “majuidn”...
3,https://turnbackhoax.id/2025/09/30/penipuan-pl...,[PENIPUAN] PLN Tawarkan Listrik Gratis Lewat A...,,tim pemeriksa fakta mafindo (turnbackhoax) men...,Tim Pemeriksa Fakta Mafindo (TurnBackHoax) men...
4,https://turnbackhoax.id/2025/09/30/salah-ojol-...,[SALAH] Ojol Demo Kenaikan Pertalite,,[salah] ojol demo kenaikan pertalite – turnbac...,Beredar video [arsip] dari akun tiktok “kampun...
5,https://turnbackhoax.id/2025/09/30/salah-pangl...,[SALAH] Panglima NATO akan Buat Kerusuhan di I...,,[salah] panglima nato akan buat kerusuhan di i...,Akun TikTok “pasukankhususalmadi7” pada Senin ...
6,https://turnbackhoax.id/2025/09/30/salah-ojol-...,[SALAH] Ojol Dilarang Gunakan Pertalite,,[salah] ojol dilarang gunakan pertalite – turn...,Pada Sabtu (20/9/2025) akun TikTok “Mr.J-Skull...
7,https://turnbackhoax.id/2025/09/30/salah-video...,[SALAH] Video Pengiriman 18.000 Personel TNI k...,,[salah] video pengiriman 18,Akun Facebook “Muchtadil Hebat Anwar” pada Sab...
8,https://turnbackhoax.id/2025/09/30/salah-pom-b...,[SALAH] Pom Bensin Shell Resmi Tutup dan Hengk...,,[salah] pom bensin shell resmi tutup dan hengk...,Beredar unggahan [arsip] dari akun Facebook “K...
9,https://turnbackhoax.id/2025/09/29/salah-mente...,[SALAH] Menteri ESDM Bahlil Lahadalia Jadi Ter...,,[salah] menteri esdm bahlil lahadalia jadi ter...,Akun Facebook “Bima Silhoute” pada Jumat (26 /...


**Mengecek File Output dan Mengunduh Dataset Hasil Scraping**

In [ ]:
# cek file di /content
import os
print("files in output:", os.listdir("/content/turnbackhoax_data")[:30])
# download
from google.colab import files
files.download("/content/turnbackhoax_data/turnbackhoax_data.csv")


files in output: ['raw_html', 'turnbackhoax_data.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Memuat Dataset Detik & TurnBackHoax serta Mengecek Struktur Awal**

In [ ]:
import pandas as pd

# Membaca kedua file
detik = pd.read_csv("detik_non_hoax_full Scraping.csv")
turnback = pd.read_csv("turnbackhoax_data (2).csv")

# Menampilkan struktur dasar
print("Detik:", detik.shape)
print("TurnBackHoax:", turnback.shape)


Detik: (3893, 3)
TurnBackHoax: (2900, 5)


# **B. Load & Pra-Pemrosesan Data**

In [ ]:
import pandas as pd
import re

# =========================================
# 1. Membaca dataset
# =========================================
turnbackhoax_path = "turnbackhoax_data (2).csv"
detik_path = "detik_non_hoax_full Scraping.csv"

df_turnbackhoax = pd.read_csv(turnbackhoax_path, encoding="utf-8")
df_detik = pd.read_csv(detik_path, encoding="utf-8")

print("Data Turnbackhoax:", df_turnbackhoax.shape)
print("Data Detik Non-hoax:", df_detik.shape)

# =========================================
# 2. Seleksi hanya berita hoaks
# =========================================
mask_hoax = df_turnbackhoax["title"].str.contains(
    "HOAKS|SALAH|DISINFORMASI|MISLEADING|FALSE|FABRICATED|PENIPUAN",
    case=False, na=False
)
df_turnbackhoax = df_turnbackhoax[mask_hoax]

print("Data Turnbackhoax setelah seleksi hoaks:", df_turnbackhoax.shape)

# =========================================
# 3. Bersihkan judul (untuk konsistensi, meski nanti tidak dipakai)
# =========================================
def clean_title(title):
    title = re.sub(r'\[(HOAKS|SALAH|DISINFORMASI|MISLEADING|FALSE|FABRICATED|PENIPUAN)\]', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s+', ' ', title).strip()
    return title

df_turnbackhoax["title"] = df_turnbackhoax["title"].apply(clean_title)

# =========================================
# 4. Pilih hanya kolom ISI berita (discard judul)
# =========================================
df_turnbackhoax = df_turnbackhoax[["content"]].rename(columns={"content": "isi"})
df_detik = df_detik[["isi_berita"]].rename(columns={"isi_berita": "isi"})

# =========================================
# 5. Tambah label
# =========================================
df_turnbackhoax["label"] = "hoaks"
df_detik["label"] = "non-hoaks"

# =========================================
# 6. Gabungkan dataset
# =========================================
df_combined = pd.concat([df_turnbackhoax, df_detik], ignore_index=True)

# =========================================
# 7. Hapus duplikat & missing
# =========================================
df_combined.drop_duplicates(subset=["isi"], inplace=True)
df_combined.dropna(subset=["isi"], inplace=True)

# =========================================
# 8. Cek hasil
# =========================================
print("\nJumlah total data:", len(df_combined))
print("\nDistribusi label:")
print(df_combined["label"].value_counts())

print("\nContoh data:")
print(df_combined.sample(5))

# =========================================
# 9. Simpan final dataset
# =========================================
output_path = "dataset_hoax_vs_nonhoax_only_isi.csv"
df_combined.to_csv(output_path, index=False, encoding="utf-8")
print(f"\n✅ Dataset final disimpan ke: {output_path}")


Data Turnbackhoax: (2900, 5)
Data Detik Non-hoax: (3893, 3)
Data Turnbackhoax setelah seleksi hoaks: (2870, 5)

Jumlah total data: 6760

Distribusi label:
label
non-hoaks    3891
hoaks        2869
Name: count, dtype: int64

Contoh data:
                                                    isi      label
6384  Tiket MotoGP Mandalika di Lombok Timur, Nusa T...  non-hoaks
1911  Hasil periksa fakta Moch. MarcellodiansyahFakt...      hoaks
2031  Hasil Periksa Fakta Dyah FegriyaniVideo terseb...      hoaks
1998  Hasil periksa fakta Rahmah.Kemendikbud memasti...      hoaks
409   Hasil periksa fakta Yudho ArdiInformasi yang m...      hoaks

✅ Dataset final disimpan ke: dataset_hoax_vs_nonhoax_only_isi.csv


# **C. PreProcessing data**

**0. Load Dataset**

In [ ]:
import pandas as pd

# Membaca dataset gabungan
df = pd.read_csv("dataset_hoax_vs_nonhoax_only_isi.csv")

# Menampilkan 5 data teratas
print("Contoh data sebelum preprocessing:")
print(df.head())


**1. Case Folding pada Kolom Isi Berita (Normalisasi Huruf Kecil)**

In [ ]:
import pandas as pd

# 1. Load dataset gabungan
df = pd.read_csv("dataset_hoax_vs_nonhoax_only_isi.csv")

# 2. Case Folding hanya pada kolom ISI
df["isi_casefold"] = df["isi"].astype(str).str.lower()

# 3. Tampilkan contoh hasil
print("Sebelum Case Folding:\n", df["isi"].iloc[0])
print("\nSesudah Case Folding:\n", df["isi_casefold"].iloc[0])


**2. Text Cleaning (Noise Removal)**

In [ ]:
import re

def clean_text(text):
    # hapus tag label hoax di awal kalimat
    text = re.sub(r"\[(hoax|salah|disinformasi|misleading|false|fabricated|penipuan)\]", "", text, flags=re.IGNORECASE)

    # hapus URL
    text = re.sub(r"http\S+|www\S+", "", text)

    # hapus angka, simbol, tanda baca
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # hapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()

    return text

# proses hanya kolom isi
df["isi_bersih"] = df["isi_casefold"].apply(clean_text)

print("Sebelum Cleaning:\n", df["isi_casefold"].iloc[0])
print("\nSesudah Cleaning:\n", df["isi_bersih"].iloc[0])


**3. Tokenisasi Teks: Pemecahan Kalimat menjadi Token Kata (NLTK Word Tokenizer)**

In [ ]:
pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.4 MB/s eta 0:00:00


In [ ]:
import nltk
from nltk.tokenize import word_tokenize

# Download tokenizer NLTK (wajib dua ini untuk versi terbaru)
nltk.download('punkt')
nltk.download('punkt_tab')

# Tokenisasi hanya pada kolom isi
df["token_isi"] = df["isi_bersih"].apply(word_tokenize)

# Contoh output
print("Sebelum Tokenisasi (isi):")
print(df["isi_bersih"].iloc[0])

print("\nSesudah Tokenisasi (isi):")
print(df["token_isi"].iloc[0])


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Sebelum Tokenisasi (isi):
faktanya belum ada dukungan resmi anies untuk pasangan manapun di pilkada jakarta unggahan berisi informasi anies baswedan mendukung calon gubernur dan wakil gubernur nomor urut pramono rano merupakan konten yang menyesatkan misleading content akun instagram lintang son o pada sabtu mengunggahvideo arsip yang memperlihatkan anies mengacungkan jari unggahan disertai takarir anies baswedan mendukung calon gubernur dan wakil gubernur nomor urut mas pram bang doelper rabu konten dilihat hampir kali dan disukai akun pemeriksaan faktadisadur dari artikel periksa fakta tirto id tim pemeriksa fakta tirto melakukan penelusuran dengan membuka media sosial anies melalui akun instagramnya aniesbaswedan terdapat foto anies yang sedang memakai batik dan mengendarai sepeda pada unggahan rabu dalam narasi yang dibagikan anies pada saat itu hanya memperingati hari batik nasional dan hari sepeda nasional bersama kelompok bike to work indonesia tirto melanjutkan penelusuran deng

**4. Stopword Removal: Menghapus Kata Umum Bahasa Indonesia (NLTK Stopwords)**

In [ ]:
import nltk
from nltk.corpus import stopwords

# Download stopwords if belum ada
nltk.download('stopwords')

# Load stopwords bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

# Hapus stopword pada kolom isi
df["token_isi_tanpa_stopword"] = df["token_isi"].apply(
    lambda tokens: [w for w in tokens if w not in stop_words]
)

# Tampilkan contoh hasil
print("Sebelum Stopword Removal:", df["token_isi"].iloc[0][:20])
print("Sesudah Stopword Removal:", df["token_isi_tanpa_stopword"].iloc[0][:20])


Sebelum Stopword Removal: ['faktanya', 'belum', 'ada', 'dukungan', 'resmi', 'anies', 'untuk', 'pasangan', 'manapun', 'di', 'pilkada', 'jakarta', 'unggahan', 'berisi', 'informasi', 'anies', 'baswedan', 'mendukung', 'calon', 'gubernur']
Sesudah Stopword Removal: ['faktanya', 'dukungan', 'resmi', 'anies', 'pasangan', 'manapun', 'pilkada', 'jakarta', 'unggahan', 'berisi', 'informasi', 'anies', 'baswedan', 'mendukung', 'calon', 'gubernur', 'wakil', 'gubernur', 'nomor', 'urut']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**5. Stemming Bahasa Indonesia Menggunakan Library Sastrawi**

In [ ]:


  from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Membuat stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Stemming pada kolom isi (hasil token tanpa stopword)
df["stem_isi"] = df["token_isi_tanpa_stopword"].apply(lambda tokens: [stemmer.stem(w) for w in tokens])

# Contoh output
print("Sebelum Stemming (isi):", df["token_isi_tanpa_stopword"].iloc[0][:10])
print("Sesudah Stemming (isi):", df["stem_isi"].iloc[0][:10])


Sebelum Stemming (isi): ['faktanya', 'dukungan', 'resmi', 'anies', 'pasangan', 'manapun', 'pilkada', 'jakarta', 'unggahan', 'berisi']
Sesudah Stemming (isi): ['fakta', 'dukung', 'resmi', 'anies', 'pasang', 'mana', 'pilkada', 'jakarta', 'unggah', 'isi']


**6. Menggabungkan Token Hasil Stemming Menjadi Kalimat Utuh**

In [ ]:
# Gabungkan kembali token menjadi kalimat utuh
df["teks_final"] = df["stem_isi"].apply(lambda x: " ".join(x))

print("Contoh Teks Final:", df["teks_final"].iloc[0])


Contoh Teks Final: fakta dukung resmi anies pasang mana pilkada jakarta unggah isi informasi anies baswedan dukung calon gubernur wakil gubernur nomor urut pramono rano konten sesat misleading content akun instagram lintang son o sabtu mengunggahvideo arsip anies acung jari unggah serta takarir anies baswedan dukung calon gubernur wakil gubernur nomor urut mas pram bang doelper rabu konten kali suka akun periksa faktadisadur artikel periksa fakta tirto id tim periksa fakta tirto telusur buka media sosial anies akun instagramnya aniesbaswedan foto anies pakai batik kendara sepeda unggah rabu narasi bagi anies ingat batik nasional sepeda nasional kelompok bike to work indonesia tirto lanjut telusur kunci dukung anies pilkada jakarta google hasil salah artikel tempo judul jubir anies baswedan nyata dukung pilkada jakarta juru bicara anies sahrin hamid anies dukung resmi satu pasang calon gubernur wakil gubernur pilkada jakarta jumat pihak rakyat lemah perhati mas anies tentu pilih tunggu 

**7. Menyimpan Dataset Hasil Preprocessing ke File CSV**

In [ ]:
df.to_csv("dataset_preprocessed_final.csv", index=False, encoding="utf-8")
print("✅ Dataset hasil preprocessing disimpan ke: dataset_preprocessed_final.csv")


✅ Dataset hasil preprocessing disimpan ke: dataset_preprocessed_final.csv


**8. Encoding Label: Konversi Hoaks (1) dan Non-Hoaks (0)bold text**

In [ ]:
import pandas as pd

# 1️⃣ Load dataset hasil preprocessing (tahap sebelumnya)
df = pd.read_csv("dataset_preprocessed_final.csv")

# 2️⃣ Encoding label (ubah hoaks → 1, non-hoaks → 0)
df["label_num"] = df["label"].map({"non-hoaks": 0, "hoaks": 1})

# 3️⃣ Cek keberhasilan encoding
print("\n✅ Contoh Encoding Label:")
print(df[["label", "label_num"]].head(10))

print("\n📊 Jumlah data per kelas:")
print(df["label_num"].value_counts())

# 4️⃣ Simpan dataset final setelah encoding label
output_path = "dataset_ready_for_feature_extraction.csv"
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"\n💾 Dataset final disimpan sebagai: {output_path}")



✅ Contoh Encoding Label:
   label  label_num
0  hoaks          1
1  hoaks          1
2  hoaks          1
3  hoaks          1
4  hoaks          1
5  hoaks          1
6  hoaks          1
7  hoaks          1
8  hoaks          1
9  hoaks          1

📊 Jumlah data per kelas:
label_num
0    3891
1    2869
Name: count, dtype: int64

💾 Dataset final disimpan sebagai: dataset_ready_for_feature_extraction.csv


**9. Exploratory View: Sampel Kolom teks_final dan label_num**

In [ ]:
import pandas as pd

# Load dataset final yang sudah kamu upload
df = pd.read_csv("dataset_ready_for_feature_extraction.csv")

# Tampilkan 20 baris pertama kolom teks_final dan label_num
df[["teks_final", "label_num"]].head(20)


,teks_final,label_num
0,fakta dukung resmi anies pasang mana pilkada j...,1
1,sumber valid benar presiden prabowo ancam nara...,1
2,nyata menkes budi lockdown great reset narasi ...,1
3,hasil cari google arah pemberitaancnbc indones...,1
4,kpk bukti kuat libat pramono puan korupsi e kt...,1
5,warganet periksa media sosial milik king kevin...,1
6,akun instagram resmi ditjenpajakri djp modus t...,1
7,lembaga kredibel rilis hasil survei tanggal in...,1
8,fakta temu informasi sumber kredibel pks mundu...,1
9,taut unggah edar arah tipu akun facebook dafta...,1
